In [1]:
import pandas as pd
import json
import requests
import time
from constants import API_KEY, TELE_TOKEN
import folium
import sys
from tqdm.notebook import tqdm
from scipy.spatial.distance import cdist
import numpy as np
import os
from datetime import date


In [ ]:
rainday = []

s = requests.Session()
s.headers.update({'x-api-key': API_KEY})
paginationToken = ""

n=1
latest_only = True
today = date.today().isoformat()

while True:
  url = f"https://api-open.data.gov.sg/v2/real-time/api/rainfall?date={today}"
  if paginationToken:
    url += ("&paginationToken=" + paginationToken)
  res = s.get(url = url)
  res_json = res.json()['data']
  rainday.append(res_json)
  if 'paginationToken' not in res_json.keys() or latest_only: break
  paginationToken = res_json['paginationToken']
  n+=1

print(f"{n} pages printed")

1 pages printed


In [3]:
station_data = rainday[0]['stations']

In [4]:
# 1. LOAD DATA
print("Reading file...")
with open('NationalMapLine.geojson', 'r') as f:
    raw_data = json.load(f)

# 2. FILTERING
filtered_features = []
exit_points = []
trunks = []
for feature in tqdm(raw_data['features'], desc="Filtering Road Hierarchy"):
    sid = feature['properties'].get('SYMBOLID')
    # Keep only Expressway (1), Slip (2), and Major Road (3)
    if sid in [1, 2, 3]:
        filtered_features.append(feature)
        if sid == 1:
            geom = feature['geometry']
            trunks.append({'road_name': feature['properties'].get('NAME'), 'coords': geom['coordinates']})
        elif sid == 2:
            geom = feature['geometry']
            if geom['type'] == 'LineString' and "expressway" not in feature['properties'].get('NAME').lower():
                # First coordinate pair in the line [longitude, latitude]
                start_coord = geom['coordinates'][0]
                exit_points.append({
                    'name': feature['properties'].get('NAME', 'Unknown Exit'),
                    'loc': [start_coord[1], start_coord[0]] # Flip to [lat, lon]
                })
    

# Create the final object
filtered_geojson = {
    "type": "FeatureCollection",
    "features": filtered_features
}

# 3. SIZE CHECK
# We convert to string to get a realistic estimate of the final HTML payload size
estimated_size_bytes = sys.getsizeof(json.dumps(filtered_geojson))
estimated_size_mb = estimated_size_bytes / (1024 * 1024)

print(f"--- Data Summary ---")
print(f"Filtered features: {len(filtered_features)}")
print(f"Estimated payload size: {estimated_size_mb:.2f} MB")
print(f"--------------------")

del raw_data

Reading file...


Filtering Road Hierarchy:   0%|          | 0/16889 [00:00<?, ?it/s]

--- Data Summary ---
Filtered features: 9165
Estimated payload size: 3.68 MB
--------------------


In [ ]:
# 4. SAFETY GATE
if estimated_size_mb > 100:
    print(f"❌ ABORTED: Filtered data is {estimated_size_mb:.2f}MB (over the 100MB safety limit).")
    print("Consider removing 'Major Roads' (SYMBOLID 3) or sampling the data.")
else:
    print("✅ Size is safe. Proceeding to render map...")
    
    # INITIALIZE MAP
    m = folium.Map(location=[1.3521, 103.8198], zoom_start=12, tiles='cartodbpositron')

    # ADD FILTERED ROADS
    folium.GeoJson(
        filtered_geojson,
        name="Filtered Roads",
        style_function=lambda x: {
            'color': '#e74c3c' if x['properties']['SYMBOLID'] == 1 else '#34495e',
            'weight': 3 if x['properties']['SYMBOLID'] == 1 else 1,
            'opacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=['NAME'])
    ).add_to(m)

    exit_group = folium.FeatureGroup(name="Expressway Exits").add_to(m)
    for ep in tqdm(exit_points, desc="Marking Exits"):
        folium.CircleMarker(
            location=ep['loc'],
            radius=2,
            color='#27ae60', # Green
            fill=True,
            fill_color='#2ecc71',
            fill_opacity=1,
            tooltip=f"Exit: {ep['name']}"
        ).add_to(exit_group)

    # ADD STATIONS
    for station in tqdm(station_data, desc="Adding Stations"):
        folium.CircleMarker(
            location=[station['location']['latitude'], station['location']['longitude']],
            radius=5,
            color='blue',
            fill=True,
            popup=f"{station['name'] + " " + station['id']}"
        ).add_to(m)

    print("Map generated below:")
    display(m) # Explicitly calling display for Jupyter

✅ Size is safe. Proceeding to render map...


Marking Exits:   0%|          | 0/543 [00:00<?, ?it/s]

Adding Stations:   0%|          | 0/77 [00:00<?, ?it/s]

Map generated below:


In [18]:
exits_df = pd.DataFrame(exit_points)
exits_df.drop_duplicates(subset='name',keep='first',inplace=True)
exits_df.head()

,name,loc
0,LOWER DELTA ROAD,"[1.2790791117483364, 103.82395017486955]"
1,YIO CHU KANG ROAD,"[1.3956192201411326, 103.85730252081304]"
4,KEPPEL ROAD,"[1.2718960747768324, 103.83150678471749]"
8,EAST COAST PARKWAY,"[1.2881563631396604, 103.86152452108897]"
11,OPHIR ROAD,"[1.2950057108297515, 103.86134876514812]"


In [20]:
def find_parent_road(exit_coords, trunks):
    best_road = "Unknown"
    min_dist = float('inf')
    
    for trunk in trunks:
        # Check distance to all points along the trunk line to find the closest segment
        trunk_coords = [[c[1], c[0]] for c in trunk['coords']] # Flip to [lat, lon]
        dists = cdist([exit_coords], trunk_coords)
        current_min = np.min(dists)
        
        if current_min < min_dist:
            min_dist = current_min
            best_road = trunk['road_name']
            
    return best_road

# Apply the calculation to generate the 'road' column
exits_df['road'] = exits_df['loc'].apply(lambda x: find_parent_road(x, trunks))

In [24]:
from scipy.spatial import distance

def get_nearest_station(exit_coords, stations):
    # distance.euclidean between exit and all stations
    dists = [
        distance.euclidean(exit_coords, [s['location']['latitude'], s['location']['longitude']])
        for s in stations
    ]
    return stations[np.argmin(dists)]['id']

exits_df['primary_station_id'] = exits_df['loc'].apply(
    lambda x: get_nearest_station(x, station_data)
)

In [26]:
from scipy.cluster.hierarchy import fcluster, linkage

# 1. Cluster nearby exits (within ~300m) to create 'Nodes'
coords_matrix = np.array(list(exits_df['loc']))
Z = linkage(coords_matrix, method='complete')
exits_df['cluster_id'] = fcluster(Z, t=0.0025, criterion='distance')

# 2. Merge names (The Slash Logic)
nodes = []
for _, group in exits_df.groupby(['road', 'cluster_id']):
    nodes.append({
        'road': group['road'].iloc[0],
        'slashed_name': " / ".join(sorted(group['name'].unique())),
        'lon': group['loc'].apply(lambda x: x[1]).mean(),
        'st_id': group['primary_station_id'].iloc[0]
    })

# 3. Sequence into Segments
master_segments = []
nodes_df = pd.DataFrame(nodes)

for road, group in nodes_df.groupby('road'):
    # Sort West to East
    sorted_nodes = group.sort_values('lon').to_dict('records')
    
    for i in range(len(sorted_nodes) - 1):
        master_segments.append({
            'road': road,
            'segment_name': f"{sorted_nodes[i]['slashed_name']} to {sorted_nodes[i+1]['slashed_name']}",
            'st_a': sorted_nodes[i]['st_id'],
            'st_b': sorted_nodes[i+1]['st_id']
        })

lookup_table = pd.DataFrame(master_segments)

In [32]:
lookup_table.to_csv("lookup.csv",index=False)